# 破解 CRC32 ― 从哈希值还原原始数据

## 本章要点

CRC32 是用于检查数据完整性的哈希，但不是密码学上的哈希。**只要知道输出，就能用计算构造出满足条件的输入**。

- 用 Z3 的 `BitVec` 把 CRC32 的计算原样写出来
- 把「以这个字符串开头」「只有英文字母」「CRC32 等于这个值」变成约束条件
- 让 Z3 造出满足条件的数据

工具是 `BitVec` 的位运算（`>>`, `&`, `^`, `~`）和 `ZeroExt`（扩展位宽）。

## 什么是 CRC32

CRC32（Cyclic Redundancy Check 32-bit）是用于检测数据错误的 32 位校验和。广泛用于网络通信和文件的完整性检查。

不过，CRC32 **不是密码学安全的哈希**。只要知道一个输出，就能反推出产生这个输出的输入。本章可以让你亲身体会到，为什么它不能用来保存密码。

## CRC32 的计算方法

CRC32 的计算可以用下面这段 C 代码表示。

```c
static uint32_t crc32(uint32_t crc, unsigned char *buf, size_t len)
{
    int k;

    crc = ~crc;
    while (len--) {
        crc ^= *buf++;
        for (k = 0; k < 8; k++)
            crc = crc & 1 ? (crc >> 1) ^ 0xedb88320 : crc >> 1;
    }
    return ~crc;
}
```

它做的事情有 4 件。

1. 把初始值按位取反（`crc = ~crc`）
2. 把每个字节依次 XOR 进去
3. 做 8 次：每次右移 1 位，如果最低位是 1 就 XOR 多项式 `0xedb88320`
4. 最后再按位取反后返回

这个计算在 Python 里也能原样写出。换成 Z3 的 `BitVec` 之后，**输入可以保持变量状态参与计算**。

In [1]:
from z3 import *


def z3_crc32(data, crc=0):
    """返回 data（BitVec(8) 的列表）的 CRC32，形式为 Z3 的表达式"""
    crc = ~crc                      # 反转初始值
    for c in data:
        crc ^= ZeroExt(24, c)       # 把 8 位字节扩展到 32 位再 XOR
        for _ in range(8):
            crc = If(crc & 1 == BitVecVal(1, 32),
                     LShR(crc, 1) ^ 0xedb88320,
                     LShR(crc, 1))
    return ~crc                     # 最后反转

## 用到的 Z3 函数

| 函数与运算符 | 作用 |
|---|---|
| `ZeroExt(24, c)` | 把 8 位的值在高位补 0，扩展到 32 位 |
| `BitVecVal(1, 32)` | 创建 32 位常数 1（位宽作为类型携带） |
| `LShR(crc, 1)` | 无符号右移（空出的高位补 0） |
| `If(条件, A, B)` | 条件为真时取 A，为假时取 B |
| `~crc` | 按位取反 |
| `^`, `&` | XOR 和 AND |

不用 `>>` 而用 `LShR()`，是因为 `BitVec` 有符号和无符号的区别。要和 C 的 `unsigned` 行为一致，就得用 `LShR()`。

另外 `BitVecVal()` 创建的是「固定值的 `BitVec`」。`simplify()` 是化简式子的函数，如果所有值都确定了，结果就会变成数值。

## 确认与 `binascii.crc32()` 是否一致

和 Python 标准的 `binascii.crc32()` 比较，确认实现是否正确。

In [2]:
from binascii import crc32

data = b'hello world!!'
print(crc32(data))

simplify(z3_crc32([BitVecVal(x, 8) for x in data]))

2909492980


2909492980

得到了相同的值。`data` 的每个字节都用 `BitVecVal(x, 8)` 这样的常数来计算，所以 Z3 的式子也被化简成了数值。

## 从 CRC32 反推数据

接下来是正题。让 Z3 找出满足下面条件的数据。

- 长度为 `n` 字节
- 以 `head` 开头，以 `tail` 结尾
- 中间的字符是英文字母（`a`〜`z`, `A`〜`Z`）
- CRC32 等于 `crc_code`

把数据的每个字节设成 `BitVec(f'b{i}', 8)` 变量，「CRC32 等于 `crc_code`」就原样成为约束条件。

In [3]:
def is_between(b, start, end):
    """b 在 start 到 end 的字符编码范围内"""
    return And(b >= ord(start), b <= ord(end))


def starts_with(data, string):
    """data 以 string 开头"""
    return And([b == ord(ch) for b, ch in zip(data, string)])


def ends_with(data, string):
    """data 以 string 结尾"""
    return And([b == ord(ch) for b, ch in zip(data[::-1], string[::-1])])


def break_crc32(n, head, tail, crc_code):
    """找出满足条件的 n 字节数据"""
    data = [BitVec(f'b{i:02d}', 8) for i in range(n)]

    solver = Solver()
    solver.add(starts_with(data, head))
    solver.add(ends_with(data, tail))

    # 中间的字符只能是英文字母
    for b in data[len(head):-len(tail)]:
        solver.add(Or(is_between(b, 'a', 'z'), is_between(b, 'A', 'Z')))

    solver.add(z3_crc32(data) == crc_code)

    if solver.check() != sat:
        return None

    model = solver.model()
    return bytes([model[d].as_long() for d in data])


def test(n, head, tail, crc_code):
    res = break_crc32(n, head, tail, crc_code)
    if res is None:
        print('unsat（不存在满足条件的数据）')
    else:
        print(res, hex(crc32(res)))

## 执行示例

找出以 `"hello "` 开头、以 `"!!"` 结尾的长度 20 的字符串，要求 CRC32 等于 `0x12345678`。

In [4]:
test(20, 'hello ', '!!', 0x12345678)

b'hello PWvJSuTYDVsZ!!' 0x12345678


输出后面的 `0x...` 就是所得字符串的 CRC32。请确认它与指定的 `0x12345678` 一致。这里用 `binascii.crc32()` 重新算了一遍，可以检查 Z3 的答案是否真的正确。

## 太短就解不出来

字符数减少，可以自由选择的字符也会减少。在这种状态下如果不存在满足条件的数据，就会变成 `unsat`。

In [5]:
test(13, 'hello ', '!!', 0x00000000)

unsat（不存在满足条件的数据）


`head` 是 6 个字符、`tail` 是 2 个字符，所以中间的字符只有 5 个。Z3 证明了：用这 5 个字符，不存在 CRC32 为 0 的数据。

把长度增加 1 字节试试。

In [6]:
test(14, 'hello ', '!!', 0x00000000)

b'hello zXforl!!' 0x0


只多了一个自由字符，就找到了满足条件的数据。

- 可搜索的字符越多，越容易找到满足条件的数据
- CRC32 是线性的计算，把条件原样写成约束，Z3 就能高速求解
- 所以 CRC32 能用来「检测篡改」，但不能用来「防止篡改」

## 小结

- CRC32 的计算可以用 `BitVec` 的位运算原样写出
- 8 位的字节用 `ZeroExt(24, c)` 扩展到 32 位
- 无符号右移用 `LShR()`，常数用 `BitVecVal()`
- 把条件（开头、结尾、字符种类、CRC 值）写成约束，Z3 就能造出满足条件的数据
- 自由字符太少就会 `unsat`（被证明不存在）